# Experiment 3 — XLM-R Causal Masking + Code-Switch Type Analysis

This notebook evaluates the XLM-R causal model with a **qualitative, per-token
breakdown by code-switch type** (`cs_type` field from SwitchLingua).

## Code-switch types in SwitchLingua
- **intra-sentential** — language switches within a clause or sentence (code-mixing)
- **inter-sentential** — language switches at sentence/clause boundaries
- **tag** — short tag switches (e.g. discourse markers)

## Analysis
For each selected language pair we compute:
1. Precision / Recall / F1 broken down by cs_type
2. Confidence distributions (P(switch)) for TP vs FP by cs_type
3. TP / FP / FN error counts per cs_type

This reveals which switch types are structurally harder for the model to predict.

In [ ]:
import os, sys, gc, pickle
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from codeswitch.auth import hf_login
from codeswitch.config import (
    ALL_LANGUAGE_PAIRS, BACKBONE_MODEL_DEFAULTS, TRAIN_PAIRS, ZEROSHOT_PAIRS,
    DataConfig, ModelConfig, QualitativeConfig, TrainConfig,
)
from codeswitch.data import build_datasets, make_collate_fn
from codeswitch.evaluate import evaluate, evaluate_per_pair, print_sigma_summary
from codeswitch.model import build_model
from codeswitch.qualitative import (
    build_qualitative_df,
    cstype_metrics_table,
    evaluate_per_token,
    plot_confidence_distributions,
    plot_cstype_metrics,
    plot_error_stacks,
    print_example_cases,
    print_qualitative_summary,
    run_qualitative_analysis,
    save_qualitative_json,
    CodeSwitchDatasetWithMeta,
)
from codeswitch.results_json import save_results_json
from codeswitch.trainer import (
    compute_class_weights, make_warmup_cosine_scheduler, set_seed, train_epoch,
)
from codeswitch.visualize import (
    plot_grouped_f1_bars, plot_lr_schedule, plot_per_pair_training_curves,
    plot_training_history, plot_universality,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
model_cfg = ModelConfig(
    backbone="xlmr",
    model_name="xlm-roberta-base",
    max_len=256,
    dropout=0.1,
    freeze_encoder=False,
)

data_cfg = DataConfig(
    dataset_name="Shelton1013/SwitchLingua_text",
    max_samples_per_pair=6000,
    train_ratio=0.8,
)

train_cfg = TrainConfig(
    batch_size=32,
    num_epochs=16,
    base_lr=1e-5,
    head_lr_multiplier=50.0,
    warmup_ratio=0.1,
    lambda_dur=1.0,
    seed=42,
)

# Qualitative analysis config
qual_cfg = QualitativeConfig(
    pairs_to_analyze=[
        "Korean-English",
        "German-English",
        "Chinese-English",
        "Spanish-English",
        "Arabic-English",
    ],
    n_examples=3,
    train_ratio=data_cfg.train_ratio,
    max_len=model_cfg.max_len,
)

train_pairs    = TRAIN_PAIRS
zeroshot_pairs = ZEROSHOT_PAIRS

CHECKPOINT   = "checkpoints/best_xlmr_mt.pt"   # reuse from Experiment 2 if available
DATA_PICKLE  = "data/preprocessed.pkl"
RESULTS_JSON = "results/train_xlmr_cstype.json"
QUAL_DIR     = "results/qualitative"

print(f"Pairs to analyse: {qual_cfg.pairs_to_analyze}")

## Data Loading

In [ ]:
if Path(DATA_PICKLE).exists():
    print(f"Loading cached data from {DATA_PICKLE}")
    with open(DATA_PICKLE, "rb") as f:
        all_stats = pickle.load(f)
    print(f"  Loaded {len(all_stats)} language pairs")
else:
    print("Cached data not found — running preprocessing...")
    hf_login()

    from datasets import load_dataset
    from codeswitch.data import analyze_language_pair
    from codeswitch.lid import ProductionLID

    dataset = load_dataset(data_cfg.dataset_name)
    tokenizer_pre = AutoTokenizer.from_pretrained(model_cfg.model_name)
    lid = ProductionLID(device=0 if torch.cuda.is_available() else -1)

    all_stats = {}
    for lang1, lang2 in ALL_LANGUAGE_PAIRS:
        stats = analyze_language_pair(
            dataset["train"], lang1, lang2, lid, tokenizer_pre,
            max_samples=data_cfg.max_samples_per_pair,
        )
        if stats:
            all_stats[f"{lang1}-{lang2}"] = stats

    Path(DATA_PICKLE).parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PICKLE, "wb") as f:
        pickle.dump(all_stats, f)
    print(f"✓ Saved to {DATA_PICKLE}")

## Build DataLoaders and Train (or Load Checkpoint)

In [ ]:
set_seed(train_cfg.seed)
tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_name)
collate   = make_collate_fn(tokenizer.pad_token_id)

train_ds, val_ds = build_datasets(
    all_stats, train_pairs, tokenizer,
    train_ratio=data_cfg.train_ratio, max_len=model_cfg.max_len,
)
train_loader = DataLoader(
    train_ds, batch_size=train_cfg.batch_size,
    shuffle=True,  collate_fn=collate, num_workers=train_cfg.num_workers,
)
val_loader = DataLoader(
    val_ds, batch_size=train_cfg.batch_size,
    shuffle=False, collate_fn=collate, num_workers=train_cfg.num_workers,
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
sw_w, dur_w = compute_class_weights(train_loader)
sw_criterion  = nn.CrossEntropyLoss(weight=sw_w.to(device),  ignore_index=-100)
dur_criterion = nn.CrossEntropyLoss(weight=dur_w.to(device))

model = build_model(model_cfg).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(),       "lr": train_cfg.base_lr},
        {"params": model.switch_head.parameters(),   "lr": train_cfg.head_lr},
        {"params": model.duration_head.parameters(), "lr": train_cfg.head_lr},
    ],
    weight_decay=train_cfg.weight_decay,
)
total_steps  = train_cfg.num_epochs * len(train_loader)
warmup_steps = int(total_steps * train_cfg.warmup_ratio)
scheduler    = make_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

plot_lr_schedule(
    warmup_steps, total_steps,
    base_lr=train_cfg.base_lr, head_lr=train_cfg.head_lr,
    output_path="results/xlmr_lr_schedule.png",
)

In [ ]:
history:    list  = []
best_sw_f1: float = 0.0
Path(CHECKPOINT).parent.mkdir(parents=True, exist_ok=True)

if Path(CHECKPOINT).exists():
    print(f"Checkpoint found — loading from {CHECKPOINT}")
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
else:
    for epoch in range(1, train_cfg.num_epochs + 1):
        losses  = train_epoch(model, train_loader, optimizer, device,
                              sw_criterion, dur_criterion, train_cfg, scheduler)
        metrics = evaluate(model, val_loader, device)
        pair_f1s = {k: v["switch_f1"] for k, v in evaluate_per_pair(
            model, all_stats, train_pairs, tokenizer, device,
            batch_size=train_cfg.batch_size, train_ratio=data_cfg.train_ratio,
            max_len=model_cfg.max_len,
        ).items()}
        history.append({
            "epoch": epoch, "loss_sw": losses["loss_sw"], "loss_dur": losses["loss_dur"],
            "switch_f1": metrics["switch_f1"], "duration_f1": metrics["duration_f1"],
            "pair_f1s": pair_f1s,
        })
        print(
            f"Epoch {epoch:02d} | "
            f"loss_sw={losses['loss_sw']:.4f}  loss_dur={losses['loss_dur']:.4f} | "
            f"sw_F1={metrics['switch_f1']:.4f}  dur_F1={metrics['duration_f1']:.4f}"
        )
        if metrics["switch_f1"] > best_sw_f1:
            best_sw_f1 = metrics["switch_f1"]
            torch.save(model.state_dict(), CHECKPOINT)
            print(f"  ✓ New best: {best_sw_f1:.4f}")

    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))

print("Model ready.")

## Evaluation and Per-Pair Overview

In [ ]:
eval_kwargs = dict(
    batch_size=train_cfg.batch_size,
    train_ratio=data_cfg.train_ratio,
    max_len=model_cfg.max_len,
)
final_train = evaluate_per_pair(model, all_stats, train_pairs, tokenizer, device, **eval_kwargs)
print_sigma_summary(final_train, {})

if history:
    plot_training_history(history, output_path="results/xlmr_cstype_training_history.png")
    pair_keys = [f"{l1}-{l2}" for l1, l2 in train_pairs]
    plot_per_pair_training_curves(
        history, pair_keys, output_path="results/xlmr_cstype_per_pair_curves.png"
    )

plot_universality(
    final_train, {},
    output_path="results/xlmr_cstype_universality.png",
    title="XLM-R Causal: Anticipatory F1 Across 15 Language Pairs",
)
plot_grouped_f1_bars(
    final_train,
    output_path="results/xlmr_cstype_grouped_f1.png",
    title="XLM-R Causal: Switch F1 vs Duration F1 per Pair",
)

## Qualitative CS-Type Analysis

For each pair in `qual_cfg.pairs_to_analyze`, we:
1. Build a `CodeSwitchDatasetWithMeta` (passes `cs_type` field through)
2. Run per-token inference on the validation split
3. Break down TP/FP/FN by cs_type
4. Compute precision, recall, F1 per cs_type
5. Show confidence distributions and example error cases

In [ ]:
# Run the full qualitative analysis pipeline
all_dfs = run_qualitative_analysis(
    model, all_stats,
    pairs=qual_cfg.pairs_to_analyze,
    tokenizer=tokenizer,
    device=device,
    output_dir=QUAL_DIR,
    cfg=qual_cfg,
)
print(f"\nAnalysed {len(all_dfs)} language pairs.")

## Deep Dive: Example Error Cases

For each analysed pair, show a few False Negative (missed switch) examples.

In [ ]:
for pair_key in qual_cfg.pairs_to_analyze:
    if pair_key not in all_dfs or pair_key not in all_stats:
        continue

    samples  = all_stats[pair_key]["processed_samples"]
    split_at = int(len(samples) * data_cfg.train_ratio)
    dataset  = CodeSwitchDatasetWithMeta(samples[split_at:], tokenizer, qual_cfg.max_len)

    df = all_dfs[pair_key]
    print(f"\n{'='*60}")
    print(f"{pair_key}")
    print(f"{'='*60}")
    print_example_cases(df, dataset, tokenizer, confusion_cat="FN", n=qual_cfg.n_examples)
    print_example_cases(df, dataset, tokenizer, confusion_cat="FP", n=qual_cfg.n_examples)

## Summary: CS-Type Metrics Across All Analysed Pairs

In [ ]:
import pandas as pd

summary_rows = []
for pair_key, df in all_dfs.items():
    tbl = cstype_metrics_table(df)
    tbl.insert(0, "pair", pair_key)
    summary_rows.append(tbl)

if summary_rows:
    summary = pd.concat(summary_rows, ignore_index=True)
    print("\nCS-Type F1 Summary:")
    print(summary.to_string(index=False))

## Save Results

In [ ]:
payload = {
    "backbone":      model_cfg.backbone,
    "model_name":    model_cfg.model_name,
    "history":       history,
    "train_results": final_train,
}
save_results_json(RESULTS_JSON, payload)
print(f"✓ JSON saved to {RESULTS_JSON}")